In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/AI_MoodMate/fer-2013.zip"
extract_path = "/content/fer2013"

# Extract only if not already extracted
if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

# Check folders
print(os.listdir(extract_path))


['train', 'val', 'test']


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, Add, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# --- Model Parameters ---
IMG_SIZE = 48
NUM_CLASSES = 7

# --- 1. Define the ResNet Block ---
def residual_block(x, filters, kernel_size=3, stride=1, use_conv_shortcut=False):
    """A standard ResNet residual block."""

    # Save the input tensor for the shortcut connection
    shortcut = x

    # Main Path - First Conv
    x = Conv2D(filters, kernel_size=kernel_size, strides=stride, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    # Main Path - Second Conv
    x = Conv2D(filters, kernel_size=kernel_size, strides=1, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)

    # Shortcut Connection
    if use_conv_shortcut:
        # 1x1 convolution for dimension matching (if needed due to stride or filter change)
        shortcut = Conv2D(filters, kernel_size=1, strides=stride, padding='valid', use_bias=False)(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add the shortcut to the main path output
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

# --- 2. Build the ResNet18-like Model for FER-2013 ---
def build_resnet18(input_shape, num_classes):
    """Builds a simplified ResNet18 model."""

    input_tensor = Input(shape=input_shape)

    # Initial Convolution (adjusting for grayscale 1-channel input)
    x = Conv2D(64, kernel_size=7, strides=2, padding='same', use_bias=False)(input_tensor)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=3, strides=2, padding='same')(x)

    # --- ResNet Blocks (Simplified ResNet18 Structure) ---
    # Block 1 (64 filters, 2 blocks)
    x = residual_block(x, filters=64)
    x = residual_block(x, filters=64)

    # Block 2 (128 filters, 2 blocks, 1st block uses stride=2 and conv shortcut)
    x = residual_block(x, filters=128, stride=2, use_conv_shortcut=True)
    x = residual_block(x, filters=128)

    # Block 3 (256 filters, 2 blocks, 1st block uses stride=2 and conv shortcut)
    x = residual_block(x, filters=256, stride=2, use_conv_shortcut=True)
    x = residual_block(x, filters=256)

    # Block 4 (512 filters, 2 blocks, 1st block uses stride=2 and conv shortcut)
    x = residual_block(x, filters=512, stride=2, use_conv_shortcut=True)
    x = residual_block(x, filters=512)

    # --- Final Classification Head ---
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x) # Adding dropout is critical to prevent overfitting on FER-2013

    # Output layer
    output_tensor = Dense(num_classes, activation='softmax')(x)

    # Create the final model
    model = Model(inputs=input_tensor, outputs=output_tensor, name='ResNet18_FER')
    return model

# --- 3. Instantiate and Compile the Model (CRITICAL STEP) ---
# Define the input shape: 48x48 pixels, 1 channel (grayscale)
input_shape = (IMG_SIZE, IMG_SIZE, 1)

# Instantiate the model
resnet18_model = build_resnet18(input_shape, NUM_CLASSES)

# Compile the model
# Using Adam with a slightly lower learning rate is often better for fine-tuning
resnet18_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Optional: Print the model summary to confirm the layers
resnet18_model.summary()

Model: "ResNet18_FER"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 48, 48, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 24, 24,    │      3,136 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 24, 24,    │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 24, 24,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 12, 12,    │          0 │ activation[0][0]  │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 12, 12,    │     36,864 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 12, 12,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 12, 12,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 12, 12,    │     36,864 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 12, 12,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 12, 12,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ max_pooling2d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 12, 12,    │          0 │ add[0][0]         │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 12, 12,    │     36,864 │ activation_2[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 12, 12,    │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 12, 12,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 12, 12,    │     36,864 │ activation_3[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 12, 12,    │        256 │ conv2d_4[0][0]  

 Total params: 11,183,431 (42.66 MB)

 Trainable params: 11,173,831 (42.62 MB)

 Non-trainable params: 9,600 (37.50 KB)

In [ ]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# --- CRITICAL VARIABLES: ADJUST THIS PATH ---
# 1. Base directory where your 'train', 'val', and 'test' folders are located.
#    (e.g., if your FER-2013 folder is called 'fer_data' inside your drive)
base_dir = '/content/fer2013' # <--- **UPDATE THIS**

# 2. Path variables for your specific directories
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')

# 3. Model/Data constants (make sure these match your model definition)
IMG_SIZE = 48
batch_size = 64

# --- 1. TRAINING GENERATOR (with Augmentation) ---
train_datagen = ImageDataGenerator(
    rescale=1./255, # Normalize pixel values
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale', # FER-2013 is grayscale (1 channel)
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

# --- 2. VALIDATION GENERATOR (NO Augmentation - FIXES THE NAMERROR) ---
# Validation data should only be scaled/normalized.
validation_datagen = ImageDataGenerator(rescale=1./255)

validation_generator = validation_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False # Do not shuffle validation data
)

Found 28709 images belonging to 7 classes.
Found 3589 images belonging to 7 classes.


In [ ]:
# --- STEP 1: LOCATE AND RUN THIS CELL ---

# Assuming 'train_generator' is already defined by your ImageDataGenerator.flow_from_directory
# This cell extracts the true count of samples for each class/emotion.

class_counts = train_generator.classes # Or another dictionary/list derived from the generator
num_classes = len(train_generator.class_indices)

# If your generator is defined like this:
# train_generator = train_datagen.flow_from_directory(...)
# You need to access the counts via:
import collections
class_counts = collections.Counter(train_generator.classes)
# The result will be a dictionary of {class_index: count}

print(class_counts)

Counter({np.int32(3): 7215, np.int32(4): 4965, np.int32(5): 4830, np.int32(2): 4097, np.int32(0): 3995, np.int32(6): 3171, np.int32(1): 436})


In [ ]:
# --- STEP 2: RERUN THIS CELL (After running the cell above) ---

# Calculate class weights
total_samples = sum(class_counts.values())
max_weight = max(class_counts.values())
class_weights_dict = {
    i: max_weight / class_counts[i]
    for i in range(num_classes)
}

print(class_weights_dict) # Confirm it looks correct

{0: 1.8060075093867334, 1: 16.548165137614678, 2: 1.7610446668293873, 3: 1.0, 4: 1.4531722054380665, 5: 1.4937888198757765, 6: 2.275307473982971}


In [ ]:
# Assuming you have run the model definition code and class weight calculation code
# The final training call should now work:

history = resnet18_model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    epochs=50,
    validation_data=validation_generator,  # <-- This variable is now defined!
    class_weight=class_weights_dict,
    # callbacks=[... your callbacks here ...]
)

Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


448/448 ━━━━━━━━━━━━━━━━━━━━ 62s 91ms/step - accuracy: 0.1783 - loss: 4.2865 - val_accuracy: 0.0880 - val_loss: 2.0609
Epoch 2/50
  1/448 ━━━━━━━━━━━━━━━━━━━━ 13s 30ms/step - accuracy: 0.1875 - loss: 3.7713

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


448/448 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.1875 - loss: 3.7713 - val_accuracy: 0.1360 - val_loss: 2.0265
Epoch 3/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 26s 58ms/step - accuracy: 0.2223 - loss: 3.5193 - val_accuracy: 0.3079 - val_loss: 1.7177
Epoch 4/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.1719 - loss: 3.7971 - val_accuracy: 0.3040 - val_loss: 1.7229
Epoch 5/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 26s 58ms/step - accuracy: 0.2773 - loss: 3.2852 - val_accuracy: 0.3586 - val_loss: 1.6564
Epoch 6/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.2969 - loss: 3.0980 - val_accuracy: 0.3700 - val_loss: 1.6288
Epoch 7/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 26s 59ms/step - accuracy: 0.3018 - loss: 3.1071 - val_accuracy: 0.3948 - val_loss: 1.5822
Epoch 8/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.2812 - loss: 4.1631 - val_accuracy: 0.3915 - val_loss: 1.5854
Epoch 9/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 26s 59ms/step - accuracy: 0.3241 - loss: 2.9997 - val_accuracy: 0.370

In [ ]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

# --- CRITICAL VARIABLES: ADJUST THIS PATH ---
# Assuming your FER-2013 base folder is still defined
base_dir = '/content/fer2013' # <--- **UPDATE THIS**
test_dir = os.path.join(base_dir, 'test') # This should point to your 'test' folder
IMG_SIZE = 48
batch_size = 64 # Use the same batch size

# --- 1. TEST GENERATOR (NO Augmentation) ---
# Test data should only be scaled/normalized.
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False # DO NOT shuffle test data!
)

Found 3589 images belonging to 7 classes.


In [ ]:
# --- 2. Make Predictions on the Test Set ---
# Predict the class probabilities
Y_pred = resnet18_model.predict(test_generator)

# Convert probabilities to class labels (0, 1, 2, ...)
y_pred_classes = np.argmax(Y_pred, axis=1)

# Get true labels
y_true_classes = test_generator.classes

# Get class names for the report
class_labels = list(test_generator.class_indices.keys())

# --- 3. Print Classification Report & Confusion Matrix ---
print('\n--- Classification Report (Milestone 2 Final Result) ---')
# This report will show precision/recall for ALL 7 classes, confirming the fix.
print(classification_report(y_true_classes, y_pred_classes, target_names=class_labels))

print('\n--- Confusion Matrix ---')
conf_matrix = confusion_matrix(y_true_classes, y_pred_classes)
print(conf_matrix)

57/57 ━━━━━━━━━━━━━━━━━━━━ 6s 40ms/step

--- Classification Report (Milestone 2 Final Result) ---
              precision    recall  f1-score   support

       Angry       0.29      0.51      0.37       491
     Disgust       0.12      0.65      0.20        55
        Fear       0.36      0.31      0.33       528
       Happy       0.79      0.50      0.61       879
     Neutral       0.53      0.33      0.41       626
         Sad       0.37      0.42      0.39       594
    Surprise       0.69      0.53      0.60       416

    accuracy                           0.44      3589
   macro avg       0.45      0.47      0.42      3589
weighted avg       0.52      0.44      0.46      3589


--- Confusion Matrix ---
[[252  50  55  18  37  71   8]
 [ 12  36   2   0   0   4   1]
 [118  34 165  23  31 117  40]
 [206  32  49 440  49  89  14]
 [122  79  38  33 207 127  20]
 [123  71  53  25  55 252  15]
 [ 36  10  99  16  12  24 219]]


In [ ]:
# Assuming 'resnet18_model' is your trained model object

# Save the model to a file, which completes the requirement for Milestone 2
resnet18_model.save('final_emotion_resnet18.h5')

print("Trained ResNet18 model saved as 'final_emotion_resnet18.h5'.")

Trained ResNet18 model saved as 'final_emotion_resnet18.h5'.


In [ ]:
from google.colab import files
# Use the name of the file that was saved:
files.download('final_emotion_resnet18.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

file_name = 'final_emotion_resnet18.h5'

if os.path.exists(file_name):
    file_size = os.path.getsize(file_name) / (1024 * 1024) # Convert bytes to MB
    print(f"✅ Success! The file '{file_name}' exists.")
    print(f"   Size: {file_size:.2f} MB")
else:
    print(f"❌ Error: The file '{file_name}' was not found in the current directory (/content/).")

✅ Success! The file 'final_emotion_resnet18.h5' exists.
   Size: 128.18 MB


In [ ]:
from google.colab import files
files.download('final_emotion_resnet18.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import zipfile
import os

# --- 1. FILE EXTRACTION (FIXES THE NameError and FileNotFoundError) ---

# Define the path to the ZIP file in your Google Drive
# **UPDATE THIS PATH if your drive structure is different**
zip_path = '/content/drive/MyDrive/AI_MoodMate/spotify_millsongdata.csv.zip'
extract_dir = '/content/music_data' # Temporary folder to extract the CSV

# Create the extraction directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

try:
    print(f"Attempting to extract ZIP file from: {zip_path}")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("ZIP file extracted successfully.")

    # The CSV filename inside the ZIP is 'spotify_millsongdata.csv'
    csv_path = os.path.join(extract_dir, 'spotify_millsongdata.csv')

    # Load the dataset using the extracted file's path
    df_songs = pd.read_csv(csv_path)
    print(f"Dataset loaded successfully. Total rows: {len(df_songs)}")

except FileNotFoundError:
    print(f"\nERROR: File not found at {zip_path}. Please verify your Google Drive path.")
    df_songs = pd.DataFrame()
except Exception as e:
    print(f"\nERROR during extraction or loading: {e}")
    df_songs = pd.DataFrame()


# --- 2. DATA CLEANING & PREPROCESSING (Runs only if data loaded successfully) ---
if not df_songs.empty:

    # --- Cleaning ---
    df_songs.dropna(subset=['text'], inplace=True)
    df_songs.drop_duplicates(subset=['artist', 'song'], inplace=True)

    def clean_text(text):
        text = re.sub(r'\r\n', ' ', text).lower()
        text = re.sub(r'[^a-z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    df_songs['clean_lyrics'] = df_songs['text'].apply(clean_text)

    # --- TF-IDF Setup on Lyrics ---
    tfidf_lyrics = TfidfVectorizer(
        stop_words='english',
        max_features=5000
    )

    tfidf_matrix_lyrics = tfidf_lyrics.fit_transform(df_songs['clean_lyrics'])

    print("\nSong Data Preprocessing Complete.")
    print(f"Total unique songs available for recommendation: {len(df_songs)}")
    print(f"TF-IDF Matrix Shape (Songs x Words): {tfidf_matrix_lyrics.shape}")


    # --- 3. EMOTION-TO-SENTENCE MAPPING & COSINE SIMILARITY FUNCTION ---

    emotion_to_sentence = {
        0: 'songs about anger, rage, conflict, frustration, and intense emotion',
        1: 'songs about sickness, hate, avoidance, repulsive feelings, and darkness',
        2: 'songs about fear, anxiety, suspense, and being trapped, worried, or nervous',
        3: 'songs about joy, fun, love, dancing, celebrating life, and feeling cheerful',
        4: 'songs about sorrow, crying, heartache, breaking up, and despair, feeling blue',
        5: 'songs about surprise, shock, excitement, sudden events, and awe',
        6: 'songs about quiet, calm, routine, everyday life, and peace, feeling steady'
    }
    emotion_index_to_name = {
        0: 'Angry', 1: 'Disgust', 2: 'Fear',
        3: 'Happy', 4: 'Sad', 5: 'Surprise', 6: 'Neutral'
    }

    def get_recommendations_lyrics(predicted_emotion_index, df_songs, tfidf_matrix, tfidf_model, top_n=5):
        if df_songs.empty:
            return "Error: Song data not loaded."

        emotion_name = emotion_index_to_name.get(predicted_emotion_index, 'Neutral')
        target_sentence = emotion_to_sentence[predicted_emotion_index]

        print(f"\n--- Detected Emotion: {emotion_name} ---")

        cleaned_sentence = clean_text(target_sentence)
        mood_vector = tfidf_model.transform([cleaned_sentence])

        cosine_sim = cosine_similarity(mood_vector, tfidf_matrix).flatten()

        song_indices = cosine_sim.argsort()[::-1]
        top_n_indices = song_indices[:top_n]

        recommendations = df_songs.iloc[top_n_indices].copy()
        recommendations['Similarity_Score'] = cosine_sim[top_n_indices]

        return recommendations[['artist', 'song', 'Similarity_Score']]


    # --- 4. DEMO THE FUNCTIONALITY (Milestone 3 Complete) ---
    print("\n*** Milestone 3: Testing Lyrical Recommendation ***")

    # Test 1: Predicts 'Happy' (Index 3)
    recommendations_happy = get_recommendations_lyrics(3, df_songs, tfidf_matrix_lyrics, tfidf_lyrics)
    print("\n**RECOMMENDATION FOR HAPPY (Index 3):**")
    print(recommendations_happy)

Attempting to extract ZIP file from: /content/drive/MyDrive/AI_MoodMate/spotify_millsongdata.csv.zip
ZIP file extracted successfully.
Dataset loaded successfully. Total rows: 57650

Song Data Preprocessing Complete.
Total unique songs available for recommendation: 57648
TF-IDF Matrix Shape (Songs x Words): (57648, 5000)

*** Milestone 3: Testing Lyrical Recommendation ***

--- Detected Emotion: Happy ---

**RECOMMENDATION FOR HAPPY (Index 3):**
                 artist               song  Similarity_Score
29747        Diana Ross           Floy Joy          0.421654
34873  Guided By Voices       Jupiter Spin          0.420094
56267   Whitney Houston                Joy          0.409997
16251    Point Of Grace  How Great Our Joy          0.406804
34916     Guns N' Roses       Ain't It Fun          0.399896


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model

# Load the model saved in Milestone 2
try:
    # Ensure this file is accessible (it should be in your default Colab directory)
    resnet18_model = load_model('final_emotion_resnet18.h5')
    print("✅ Emotion model 'final_emotion_resnet18.h5' loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")

✅ Emotion model 'final_emotion_resnet18.h5' loaded successfully.


In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
# The following objects are assumed to be defined from Milestone 3:
# df_songs, tfidf_matrix_lyrics, tfidf_lyrics, clean_text, get_recommendations_lyrics

# Define constants
IMG_SIZE = 48
emotion_index_to_name = {
    0: 'Angry', 1: 'Disgust', 2: 'Fear',
    3: 'Happy', 4: 'Sad', 5: 'Surprise', 6: 'Neutral'
}

def moodmate_recommend(image_path):
    # ... (Your function code here) ...
    if 'resnet18_model' not in globals():
        print("ERROR: Emotion model is not loaded. Please run the model loading cell first.")
        return

    # --- STEP 1: Load and Preprocess Image ---
    try:
        img = image.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE), color_mode='grayscale')
    except FileNotFoundError:
        print(f"ERROR: Image file not found at path: {image_path}")
        return

    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0 # Rescale

    # --- STEP 2: Predict Emotion ---
    predictions = resnet18_model.predict(img_array, verbose=0)
    predicted_index = np.argmax(predictions[0])
    predicted_emotion = emotion_index_to_name[predicted_index]

    # --- STEP 3: Generate Music Recommendations ---
    recommendations = get_recommendations_lyrics(
        predicted_index,
        df_songs,
        tfidf_matrix_lyrics,
        tfidf_lyrics
    )

    # --- STEP 4: Print Final Result ---
    print("\n=====================================================")
    print(f"✅ FINAL RESULT: Detected Emotion is: {predicted_emotion.upper()}")
    print("=====================================================")
    print("🎶 Recommended Songs for Your Mood (Lyrical Match):")

    if isinstance(recommendations, pd.DataFrame):
        for index, row in recommendations.iterrows():
            print(f"- {row['artist']} - {row['song']} (Score: {row['Similarity_Score']:.4f})")
    else:
        print(recommendations)

    return predicted_emotion, recommendations

In [ ]:
# --- STEP 1: DEFINE YOUR IMAGE PATH ---
# IMPORTANT: REPLACE THE DUMMY PATH BELOW with a valid path to an image file
# (e.g., a 'happy', 'sad', or 'angry' image from your FER-2013 val/test set).

test_image_path = '/content/fer2013/test/Happy/34746.png'

# --- STEP 2: CALL THE FUNCTION ---
print(f"\nAttempting end-to-end prediction for image: {test_image_path}")
print("-----------------------------------------------------")

# This single line runs your entire project: CNN prediction + TF-IDF recommendation
final_emotion, final_songs = moodmate_recommend(test_image_path)


Attempting end-to-end prediction for image: /content/fer2013/test/Happy/34746.png
-----------------------------------------------------

--- Detected Emotion: Happy ---

✅ FINAL RESULT: Detected Emotion is: HAPPY
🎶 Recommended Songs for Your Mood (Lyrical Match):
- Diana Ross - Floy Joy (Score: 0.4217)
- Guided By Voices - Jupiter Spin (Score: 0.4201)
- Whitney Houston - Joy (Score: 0.4100)
- Point Of Grace - How Great Our Joy (Score: 0.4068)
- Guns N' Roses - Ain't It Fun (Score: 0.3999)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import os

# --- 1. MODEL ARCHITECTURE (UPDATED FOR 3 INPUT CHANNELS) ---

def conv_block(x, filters, stride=1, weight_decay=1e-4):
    """Standard ResNet Residual Block structure."""
    shortcut = x

    # First Conv -> BN -> ReLU
    x = layers.Conv2D(filters, 3, strides=stride, padding='same',
                      use_bias=False, kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Second Conv -> BN
    x = layers.Conv2D(filters, 3, strides=1, padding='same',
                      use_bias=False, kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = layers.BatchNormalization()(x)

    # Shortcut connection
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same',
                                 use_bias=False, kernel_regularizer=regularizers.l2(weight_decay))(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    # Add and Final ReLU
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x

def build_resnet18(input_shape=(48,48,3), num_classes=7):
    # CRITICAL CHANGE: input_shape changed to (48,48,3) for RGB
    """Complete ResNet18 structure for 48x48 RGB input."""
    inputs = layers.Input(shape=input_shape)

    # Initial Convolution, BN, ReLU, MaxPool
    x = layers.Conv2D(64, 7, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(3, strides=2, padding='same')(x)

    # ResNet Blocks
    for filters, blocks, stride in [(64,2,1), (128,2,2), (256,2,2), (512,2,2)]:
        for b in range(blocks):
            x = conv_block(x, filters, stride if b == 0 else 1)

    # Final Classification Layers
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model

# --- 2. DATA LOADING AND TRAINING (UPDATED FOR RGB) ---

# !!! ADJUST THESE PATHS to match your Colab data location !!!
train_dir = '/content/fer2013/train'
test_dir = '/content/fer2013/test'
# !!! ------------------------------------------------ !!!

# Data Augmentation and Normalization
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Load training data
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(48, 48),
    batch_size=64,
    color_mode='rgb', # CRITICAL CHANGE: Load as RGB
    class_mode='categorical'
)

# Load validation/test data
validation_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(48, 48),
    batch_size=64,
    color_mode='rgb', # CRITICAL CHANGE: Load as RGB
    class_mode='categorical'
)

# --- MODEL COMPILATION AND TRAINING ---
model = build_resnet18()

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Use early stopping to prevent overfitting
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
]

# Train the model (You will likely need to adjust the number of epochs)
print("Starting training with RGB images...")
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.n // train_generator.batch_size,
    epochs=50, # Start with 50, but training may stop early via callbacks
    validation_data=validation_generator,
    validation_steps=validation_generator.n // validation_generator.batch_size,
    callbacks=callbacks
)

# --- SAVE THE NEW WEIGHTS (CORRECTED) ---
# Keras requires a specific filename format. We will save it correctly first.
temp_filename = 'model_weights.weights.h5'

# Save the weights using the correct filename and no unnecessary arguments
model.save_weights(temp_filename)
print(f"Weights saved temporarily as: {temp_filename}")

# Rename the file to the required name for your Hugging Face Space
os.rename(temp_filename, 'model_weights.h5')
print("File successfully renamed to: model_weights.h5")

Found 28709 images belonging to 7 classes.
Found 3589 images belonging to 7 classes.
Starting training with RGB images...
Epoch 1/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 69s 114ms/step - accuracy: 0.2511 - loss: 2.4542 - val_accuracy: 0.3069 - val_loss: 2.1281
Epoch 2/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2188 - loss: 2.1424 - val_accuracy: 0.3083 - val_loss: 2.1303
Epoch 3/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 41s 91ms/step - accuracy: 0.3286 - loss: 2.1265 - val_accuracy: 0.3993 - val_loss: 1.9681
Epoch 4/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3906 - loss: 2.0890 - val_accuracy: 0.3915 - val_loss: 1.9838
Epoch 5/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 40s 88ms/step - accuracy: 0.3695 - loss: 2.0202 - val_accuracy: 0.4520 - val_loss: 1.8332
Epoch 6/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4219 - loss: 1.8304 - val_accuracy: 0.4512 - val_loss: 1.8311
Epoch 7/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 41s 92ms/step - accuracy: 0.4137 - loss: 1.9285 - val_accuracy:

In [ ]:
# --- SAVE THE NEW WEIGHTS (CORRECTED) ---
# Keras requires a specific filename format. We will save it correctly first.
temp_filename = 'model_weights.weights.h5'

# Save the weights using the correct filename and no unnecessary arguments
model.save_weights(temp_filename)
print(f"Weights saved temporarily as: {temp_filename}")

# Rename the file to the required name for your Hugging Face Space
os.rename(temp_filename, 'model_weights.h5')
print("File successfully renamed to: model_weights.h5")

Weights saved temporarily as: model_weights.weights.h5
File successfully renamed to: model_weights.h5


In [ ]:
from google.colab import files
files.download('model_weights.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

# 1. Use a temporary file name that Keras is happy with
temp_filename = 'new_model_weights.weights.h5'

# 2. Save the weights (This should not give any errors now)
model.save_weights(temp_filename)

# 3. Rename the file to the required name for your Hugging Face Space
# This file is the one you must download!
os.rename(temp_filename, 'model_weights.h5')
print("Weights have been saved and renamed successfully. Now download 'model_weights.h5'.")

Weights have been saved and renamed successfully. Now download 'model_weights.h5'.


In [ ]:
from google.colab import files
files.download('model_weights.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

# Check the size of the file you are about to download
file_size_bytes = os.path.getsize('model_weights.h5')
file_size_mb = file_size_bytes / (1024 * 1024)

print(f"Size of 'model_weights.h5': {file_size_mb:.2f} MB")

# If the size is less than 1 MB, the save failed! You must re-run the save command.
if file_size_mb < 1.0:
    print("WARNING: File size is too small. Please re-run the save command below.")
    # Re-run the clean save logic one last time if the size is too small
    temp_filename = 'new_model_weights.weights.h5'
    model.save_weights(temp_filename)
    os.rename(temp_filename, 'model_weights.h5')

    file_size_bytes = os.path.getsize('model_weights.h5')
    file_size_mb = file_size_bytes / (1024 * 1024)
    print(f"NEW Size of 'model_weights.h5': {file_size_mb:.2f} MB")

# Download the file
from google.colab import files
files.download('model_weights.h5')

Size of 'model_weights.h5': 128.20 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model.save('full_model_rgb.h5')

In [ ]:
# Force the file download from the Colab machine to your local computer
from google.colab import files
files.download('full_model_rgb.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
# ==============================================================================
# PURE PYTHON COMMUNICATION TEST (No CV2, No TensorFlow)
# GOAL: Verify kernel communication without ANY native image processing libraries.
# ==============================================================================

import base64
import json
from IPython.display import display, Javascript, HTML
from google.colab.output import register_callback

# --- PYTHON TEST FUNCTION ---
def python_image_processor(b64_image_data):
    """
    Receives base64 image data and returns it immediately.
    """
    result_text = "Status: Communication Test Failed (Pure Python)."
    b64_reflected_data = ""

    try:
        # 1. LOG INPUT SIZE
        b64_data_only = b64_image_data.split(',')[1]
        print(f"--- PURE PYTHON TEST: Image data size received: {len(b64_data_only)/1024:.2f} KB ---")

        # 2. PERFORM NO PROCESSING (Only reflect the data)
        # We just need to ensure the base64 string is correctly extracted.
        b64_reflected_data = b64_data_only

        result_text = "PURE PYTHON TEST SUCCESS! (Image reflected back without processing)."

    except Exception as e:
        print(f"CRITICAL PYTHON EXCEPTION DURING PURE PYTHON TEST: {e}")
        result_text = f"FATAL ERROR: Communication failed during Python processing: {e}."
        b64_reflected_data = ""

    # 3. RETURN JSON (Guaranteed to be valid JSON structure)
    return json.dumps({
        "status": "success",
        "text": result_text,
        "image_data": b64_reflected_data
    })


# --- JAVASCRIPT/HTML UI SETUP (Standard) ---
def start_ui():
    """Generates the HTML/JS for the interactive interface."""

    register_callback('python_image_processor', python_image_processor)

    js_code = """
    async function processImage(dataURL) {
        // ... (JS logic for calling python_image_processor is standard)
        document.getElementById('processing-spinner').style.display = 'inline-block';
        document.getElementById('status-message').innerText = 'Status: Sending image to kernel...';

        document.getElementById('webcam-video').style.display = 'none';
        document.getElementById('result-image').style.display = 'none';

        // This is necessary because the reflected image uses the original dataURL mime type
        const mimeType = dataURL.split(';')[0].split(':')[1] || 'image/jpeg';

        let resultText = 'Error: Kernel communication failed or response was empty.';
        let imageData = '';

        try {
            const pythonResult = await google.colab.kernel.invokeFunction(
                'python_image_processor',
                [dataURL],
                {}
            );

            if (pythonResult && pythonResult.data && pythonResult.data.text) {
                const jsonString = pythonResult.data.text[0];
                const result = JSON.parse(jsonString);

                resultText = result.text;
                imageData = result.image_data;
            } else {
                 console.error("Python result structure unexpected or empty:", pythonResult);
            }

        } catch (e) {
            resultText = 'FATAL ERROR: Could not communicate with Python kernel: ' + e.message;
            console.error("Invoke Function Error:", e);
        }

        document.getElementById('status-message').innerText = resultText;
        if (imageData) {
            // Use the determined mimeType for the reflected image
            document.getElementById('result-image').src = mimeType + ';base64,' + imageData;
        }

        document.getElementById('processing-spinner').style.display = 'none';
        document.getElementById('result-image').style.display = 'block';
    }

    // --- Standard UI and Control Functions (omitted for brevity, assume they are present) ---

    let videoStream;

    function startWebcam() {
        const video = document.getElementById('webcam-video');
        document.getElementById('result-image').style.display = 'none';
        document.getElementById('webcam-video').style.display = 'block';
        document.getElementById('status-message').innerText = 'Status: Requesting camera permission...';

        if(videoStream) {
            videoStream.getTracks().forEach(track => track.stop());
        }

        navigator.mediaDevices.getUserMedia({ video: { facingMode: 'user' } })
            .then(stream => {
                videoStream = stream;
                video.srcObject = stream;
                document.getElementById('webcam-controls').style.display = 'flex';
                document.getElementById('upload-controls').style.display = 'flex';
                document.getElementById('status-message').innerText = 'Status: Ready to capture from webcam.';
            })
            .catch(err => {
                document.getElementById('status-message').innerText = 'ERROR: Could not start webcam. Please ensure permission is granted.';
                console.error("Webcam Error: ", err);
            });
    }

    function captureAndPredict() {
        const video = document.getElementById('webcam-video');
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0, canvas.width, canvas.height);

        if(videoStream) {
            videoStream.getTracks().forEach(track => track.stop());
        }

        const dataURL = canvas.toDataURL('image/jpeg');
        processImage(dataURL);
    }

    document.getElementById('file-upload').addEventListener('change', function(event) {
        const file = event.target.files[0];
        if (file) {
            const reader = new FileReader();
            reader.onload = function(e) {
                if(videoStream) {
                    videoStream.getTracks().forEach(track => track.stop());
                }
                processImage(e.target.result);
            };
            reader.readAsDataURL(file);
        }
    });

    document.getElementById('result-image').onload = function() {
        document.getElementById('webcam-video').style.display = 'none';
        document.getElementById('result-image').style.display = 'block';
    }

    setTimeout(startWebcam, 50);
    """

    # --- HTML UI (Standard) ---
    html_ui = """
    <style>
        .container-box {
            font-family: 'Inter', Arial, sans-serif;
            border: 1px solid #ddd;
            padding: 20px;
            border-radius: 12px;
            box-shadow: 0 8px 16px rgba(0,0,0,0.1);
            max-width: 800px;
            margin: 20px auto;
            background-color: #ffffff;
        }
        .header {
            text-align: center;
            color: #EAB308; /* Changed color to yellow/gold for diagnostic mode */
            margin-bottom: 20px;
            font-size: 1.8rem;
            font-weight: 700;
        }
        .video-feed, .result-display {
            width: 100%;
            max-height: 400px;
            object-fit: contain;
            border-radius: 8px;
            display: block;
            margin-bottom: 15px;
            box-shadow: 0 0 5px rgba(0,0,0,0.1);
        }
        #webcam-video {
            background: #222;
        }
        .controls {
            display: flex;
            flex-wrap: wrap;
            gap: 15px;
            justify-content: center;
            margin-top: 15px;
        }
        .control-group {
            display: flex;
            gap: 10px;
        }
        button, label {
            background-color: #FBBF24;
            color: black;
            padding: 10px 18px;
            border: none;
            border-radius: 6px;
            cursor: pointer;
            font-size: 15px;
            font-weight: 500;
            transition: background-color 0.2s, transform 0.1s;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            text-align: center;
            flex-grow: 1;
        }
        button:hover, label:hover {
            background-color: #EAB308;
            transform: translateY(-1px);
        }
        .spinner {
            border: 4px solid #f3f3f3;
            border-top: 4px solid #FBBF24;
            border-radius: 50%;
            width: 20px;
            height: 20px;
            animation: spin 1s linear infinite;
            display: none;
            margin-left: 10px;
            vertical-align: middle;
        }
        @keyframes spin {
            0% { transform: rotate(0deg); }
            100% { transform: rotate(360deg); }
        }
        .message-box {
            background-color: #FFFBEB;
            color: #92400E;
            padding: 12px;
            border-radius: 6px;
            border: 1px solid #FDE68A;
            min-height: 45px;
            text-align: center;
            margin-top: 20px;
            font-weight: 600;
            display: flex;
            align-items: center;
            justify-content: center;
        }
    </style>

    <div class="container-box">
        <h2 class="header">AI MoodMate: PURE PYTHON TEST</h2>

        <video id="webcam-video" autoplay playsinline class="video-feed" style="display:block;"></video>

        <img id="result-image" class="result-display" style="display:none;"/>

        <div class="message-box">
            <span id="status-message">Initializing...</span>
            <div id="processing-spinner" class="spinner"></div>
        </div>

        <div class="controls">
            <div class="control-group" id="webcam-controls" style="display:none;">
                <button onclick="captureAndPredict()">Capture & Reflect</button>
                <button onclick="startWebcam()">Restart Camera</button>
            </div>

            <div class="control-group" id="upload-controls" style="display:none;">
                <label for="file-upload">
                    Upload Image
                </label>
                <input type="file" id="file-upload" accept="image/*" style="display:none;">
            </div>
        </div>
    </div>
    """

    # 5. Run the main UI initialization
    display(HTML(html_ui), Javascript(js_code))


# --- EXECUTION ---
start_ui()

<IPython.core.display.Javascript object>

--- PURE PYTHON TEST: Image data size received: 75.36 KB ---
--- PURE PYTHON TEST: Image data size received: 2.19 KB ---
